# FitCheck 데모 — 박정우 버전 (치수를 몰라요 포함) · Colab T4

정지훈 저장소(`jhjung03eee/virtual-fitting-app`)의 앱 그대로 + **"치수를 몰라요 — 키·몸무게로 내 사이즈 찾기"** (03 섹션 아래 파란 상자).
코드는 내 포크 `jungwoopark-ai/virtual-fitting-app` 에서 받습니다. 합성 엔진은 FASHN VTON v1.5.

**먼저: 런타임 → 런타임 유형 변경 → T4 GPU.** 셀을 위에서부터 실행하면 마지막에 공개 링크(`https://....gradio.live`)가 나옵니다.

| | T4 (무료) |
|---|---|
| 설치 + 가중치 (최초 1회) | 약 5분 |
| 앱 시작 준비 운동 (torch.compile) | 약 1~2분 |
| 합성 한 장 (25스텝) | 약 50초 |
| 사이즈 추천 · 치수를 몰라요 | 1초 미만 (GPU 안 씀) |

In [ ]:
# 1. GPU 확인 — T4 가 보여야 합니다
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 코드 받기 (내 포크 = 치수를 몰라요 버전 + FASHN)
import os

APP = '/content/vfa-jungwoo'
FASHN = '/content/fashn-vton-1.5'
WEIGHTS = '/content/fashn-weights'

if not os.path.exists(APP):
    !git clone -q https://github.com/jungwoopark-ai/virtual-fitting-app.git {APP}
else:
    !git -C {APP} pull -q
if not os.path.exists(FASHN):
    !git clone -q --depth 1 https://github.com/fashn-AI/fashn-vton-1.5.git {FASHN}
!git -C {APP} log --oneline -1
!ls {APP}/app/size_unknown.py {APP}/data/body_models/
print('완료')

In [ ]:
# 3. 설치 (최초 1회 약 3분)
!pip install -q -e {FASHN}
!pip install -q mediapipe gradio scikit-image
print('완료')

In [ ]:
# 4. 가중치 내려받기 (약 2GB, 최초 1회)
if not os.path.exists(os.path.join(WEIGHTS, 'model.safetensors')):
    !python {FASHN}/scripts/download_weights.py --weights-dir {WEIGHTS}
print('완료')

In [ ]:
# 5. 데모 실행 — 출력의 "공개 링크 :" 를 새 탭으로 여세요 (처음 로딩 약 1~2분). 멈추려면 셀 중지.
os.environ['FASHN_REPO'] = FASHN
os.environ['FASHN_WEIGHTS'] = WEIGHTS
!python {APP}/app/gradio_app.py

## 화면에서 써 보기

1. **치수를 몰라요** (줄자 없이): 02에서 옷 종류(상의/하의) → 03의 옷 치수표 고르기 → 아래 파란 상자에 **성별 · 키 · 몸무게** →
   **키·몸무게로 내 사이즈 찾기**. 오른쪽 카드에 사이즈별 **맞을 확률**과 추천 사이즈가 나옵니다.
   확률이 60% 미만이면 "경계에 있어요" — 위 칸에 줄자로 잰 가슴둘레(하의는 허리둘레)만 넣고 다시 누르면 더 정확해집니다.
2. **줄자로 잰 치수가 있으면**: 03 칸에 넣고 **사이즈만 보기** (정지훈 버전 그대로).
3. **피팅 해보기**: 인물 사진(정면·전신) + 옷 사진을 올리고 누르면 약 50초 후 결과 한 장.

## 잘 안 될 때
- **공개 링크가 안 나옴**: 5번 셀 출력 아래 안내대로 `eval_js` 로 포트를 엽니다.
- **사진을 올렸는데 "다시 찍어주세요"**: 정면 · 팔은 몸에서 주먹 하나 · 머리부터 발끝까지 꽉 채운 사진이어야 합니다.
- **메모리 부족**: 런타임 다시 시작 → 2번(경로)과 5번 셀만 실행.
- 2026-11-09 Kaggle T4 에서 같은 순서로 확인: 설치·가중치 68초 → 모델 21초 → 준비 운동 59초 → 앱·공개 링크 생성 → 치수를 몰라요·사이즈만 보기 버튼 → 합성 48초.